# Model & data audit — Cordilla account scoring

Built step by step, one check at a time. See `RESEARCH-LOG.md` for the questions and hypotheses this audit is meant to inform.

## Objective

Determine whether `model/model.pkl` can be trusted to influence how reps allocate their limited attention, and whether it's worth investing further effort in it — not to retrain or improve the model, but to decide what to trust it for, what not to, and whether to ship its scores as-is.

## Audit questions

1. **Performance, overall and by segment.** How good is the model in aggregate, and does that hold evenly across account segments — or is it materially more (or less) trustworthy for some segments than others?
2. **Variable correlation to target.** Which features actually move with `converted_within_90d`, and do the relationships make sense?
3. **Is the model still useful as-is?** How old is the training data, and how does its distribution compare to `accounts_to_score.csv`? Data drift is the leading suspect for what likely killed the earlier Cordilla scoring effort, so this checks whether the same failure mode applies here.
4. **Label leakage / circularity.** Are any features (e.g. `sales_contacts_90d`, marketing engagement fields) consequences of a rep already having contacted the account, rather than independent predictors available *before* that contact decision? If so the model may be learning "who did we already talk to," not "who is likely to convert" — which would make it unusable for the attention-allocation decision it's meant to support.
5. **Selection bias in the training population.** Was `training_data.csv` a representative sample of the account universe, or already filtered by whatever selection reps or managers applied historically? If the latter, the model never learned from the accounts that were ignored, and its scores may not generalize to the full `accounts_to_score.csv` universe.
6. **Baseline comparison.** Does the model meaningfully outperform a trivial heuristic (e.g., sorting by company size or an existing engagement field)? If a simple rule captures most of the value, that changes the recommendation.
7. **Feature parity between training and scoring data.** Do all training features exist in `accounts_to_score.csv`, with the same definitions, units, and missingness patterns? A silently mismatched or missing column would break the model quietly.

## Noted limitation (not checked here)

`converted_within_90d` is a conversion count/flag, not a revenue figure — if deal sizes vary a lot, a model optimized for conversion probability could systematically undervalue high-ARR accounts. There's no deal-size field available to verify this either way, so it's flagged as an open limitation rather than something this audit can resolve.
